# SmartApply Autopilot Notebook

Notebook de pilotage pour executer SmartApply depuis Jupyter.

Ce notebook lit `.env`, initialise la base, lance l'autopilot, puis affiche les jobs, candidatures, audits et fichiers generes.

Important: par defaut `CREATE_GMAIL_DRAFTS = False` pour eviter de creer des brouillons Gmail par accident. Mets-le a `True` quand tes credentials Gmail sont prets.

## 1. Setup projet et configuration

In [5]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "smartapply").exists():
    PROJECT_ROOT = Path("/Users/nourlachtar/projets/projet/smart apply")

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from smartapply.config import get_settings
from smartapply.database import init_db, session_scope
from smartapply.database.repository import list_applications, list_jobs, total_cost

get_settings.cache_clear()
settings = get_settings()
init_db()

print(f"Project root: {PROJECT_ROOT}")
print(f"Database: {settings.database_url}")
print("DB initialized")

Project root: /Users/nourlachtar/projets/projet/smart apply
Database: sqlite:///./data/smartapply.db
DB initialized


In [2]:
def configured(value: str | None) -> str:
    return "OK" if bool(value) else "missing"

config_status = {
    "OPENAI_API_KEY": configured(settings.openai_api_key),
    "SERPAPI_API_KEY": configured(settings.serpapi_api_key),
    "FRANCETRAVAIL_CLIENT_ID": configured(settings.francetravail_client_id),
    "FRANCETRAVAIL_CLIENT_SECRET": configured(settings.francetravail_client_secret),
    "GMAIL_CREDENTIALS_PATH exists": str(Path(settings.gmail_credentials_path).exists()),
    "GMAIL_TOKEN_PATH exists": str(Path(settings.gmail_token_path).exists()),
    "ANYMAILFINDER_API_KEY": configured(settings.anymailfinder_api_key),
    "ANYMAILFINDER_MAX_CONTACTS": settings.anymailfinder_max_contacts,
    "ANYMAILFINDER_DECISION_MAKER_CATEGORIES": settings.anymailfinder_decision_maker_categories,
    "ANYMAILFINDER_COMPANY_EMAIL_TYPE": settings.anymailfinder_company_email_type,
    "CONTACT_CACHE_ENABLED": settings.contact_cache_enabled,
    "LLM_PROVIDER": settings.llm_provider,
    "EMBEDDINGS_PROVIDER": settings.embeddings_provider,
    "AUTOPILOT_TARGET_DRAFTS": settings.autopilot_target_drafts,
    "AUTOPILOT_MIN_SCORE": settings.autopilot_min_score,
    "AUTOPILOT_ANALYZE_MULTIPLIER": settings.autopilot_analyze_multiplier,
    "AUTOPILOT_CANDIDATE_MULTIPLIER": settings.autopilot_candidate_multiplier,
}
print(json.dumps(config_status, indent=2, ensure_ascii=False))

{
  "OPENAI_API_KEY": "missing",
  "SERPAPI_API_KEY": "OK",
  "FRANCETRAVAIL_CLIENT_ID": "missing",
  "FRANCETRAVAIL_CLIENT_SECRET": "missing",
  "GMAIL_CREDENTIALS_PATH exists": "False",
  "GMAIL_TOKEN_PATH exists": "False",
  "ANYMAILFINDER_API_KEY": "missing",
  "ANYMAILFINDER_MAX_CONTACTS": 5,
  "ANYMAILFINDER_DECISION_MAKER_CATEGORIES": "hr,engineering,it",
  "ANYMAILFINDER_COMPANY_EMAIL_TYPE": "generic",
  "CONTACT_CACHE_ENABLED": true,
  "LLM_PROVIDER": "openai",
  "EMBEDDINGS_PROVIDER": "openai",
  "AUTOPILOT_TARGET_DRAFTS": 25,
  "AUTOPILOT_MIN_SCORE": 0.62,
  "AUTOPILOT_ANALYZE_MULTIPLIER": 2.0,
  "AUTOPILOT_CANDIDATE_MULTIPLIER": 3.0
}


## 2. Etat actuel de la base

In [3]:
with session_scope() as s:
    jobs = list(list_jobs(s, limit=500))
    apps = list(list_applications(s))
    status_counts = {}
    for job in jobs:
        status_counts[job.status] = status_counts.get(job.status, 0) + 1
    cost = total_cost(s)

print("Jobs:", len(jobs))
print("Applications:", len(apps))
print("Status counts:", json.dumps(status_counts, indent=2))
print("LLM cost USD:", round(cost, 4))

Jobs: 0
Applications: 0
Status counts: {}
LLM cost USD: 0.0


In [4]:
try:
    import pandas as pd
    with session_scope() as s:
        rows = []
        for job in list_jobs(s, limit=50):
            rows.append({
                "id": job.id,
                "status": job.status,
                "score": job.score.final_score if job.score else None,
                "title": job.title,
                "company": job.company,
                "location": job.location,
                "source": job.source,
            })
    display(pd.DataFrame(rows))
except ImportError:
    print("pandas not installed; skipping dataframe display")

""


## 3. Lancer l'autopilot

Ajuste les variables ci-dessous puis execute la cellule. Pour un premier run prudent, garde `CREATE_GMAIL_DRAFTS = False`. Pour creer des brouillons Gmail, mets `True`.

In [ ]:
QUERY = "Data Scientist OR Machine Learning Engineer OR AI Engineer"
LOCATION = "Paris, France"
SOURCES = ["serpapi", "francetravail", "manual"]
TARGET_DRAFTS = 25
MAX_PER_SOURCE = 40
CREATE_GMAIL_DRAFTS = False
REQUIRE_QUALITY_GATE = True

print("Autopilot params:")
print(json.dumps({
    "query": QUERY,
    "location": LOCATION,
    "sources": SOURCES,
    "target_drafts": TARGET_DRAFTS,
    "max_per_source": MAX_PER_SOURCE,
    "create_gmail_drafts": CREATE_GMAIL_DRAFTS,
    "require_quality_gate": REQUIRE_QUALITY_GATE,
}, indent=2))

In [ ]:
from smartapply.jobsearch import AutopilotRunner

runner = AutopilotRunner()
report = runner.run(
    query=QUERY,
    location=LOCATION,
    sources=SOURCES,
    max_per_source=MAX_PER_SOURCE,
    target_drafts=TARGET_DRAFTS,
    create_gmail_drafts=CREATE_GMAIL_DRAFTS,
    require_quality_gate=REQUIRE_QUALITY_GATE,
)
report_dict = report.to_dict()
print(json.dumps(report_dict, indent=2, ensure_ascii=False, default=str))

In [ ]:
try:
    import pandas as pd
    rows = []
    for app in report_dict.get("applications", []):
        quality = app.get("quality_review") or {}
        rows.append({
            "job_id": app.get("job_id"),
            "application_id": app.get("application_id"),
            "status": app.get("status"),
            "contact_email": app.get("contact_email"),
            "contact_form_url": app.get("contact_form_url"),
            "gmail_draft_id": app.get("gmail_draft_id"),
            "match": quality.get("match_score"),
            "cv": quality.get("cv_score"),
            "email": quality.get("email_score"),
            "reason": quality.get("decision_reason"),
        })
    display(pd.DataFrame(rows))
except NameError:
    print("Run the autopilot cell first")
except ImportError:
    print("pandas not installed")

## 4. Inspecter les candidatures generees

In [ ]:
try:
    import pandas as pd
    with session_scope() as s:
        rows = []
        for app in list_applications(s):
            rows.append({
                "id": app.id,
                "job_id": app.job_id,
                "status": app.status,
                "company": app.job.company if app.job else None,
                "title": app.job.title if app.job else None,
                "contact": app.contact.email if app.contact else None,
                "gmail_draft_id": app.gmail_draft_id,
                "cv_docx_path": app.cv_docx_path,
                "eml_path": app.eml_path,
                "notes": app.notes,
            })
    display(pd.DataFrame(rows))
except ImportError:
    print("pandas not installed")

## 5. Voir l'audit Autopilot d'une candidature

Mets `APPLICATION_ID` sur l'id d'une candidature pour afficher le JSON d'audit.

In [ ]:
APPLICATION_ID = None

if APPLICATION_ID is None:
    print("Set APPLICATION_ID to inspect an audit")
else:
    from smartapply.database.models import Application
    with session_scope() as s:
        app = s.get(Application, int(APPLICATION_ID))
        audits = [d for d in app.documents if d.doc_type == "autopilot_audit"] if app else []
        if not audits:
            print("No autopilot audit found")
        else:
            audit = json.loads(audits[-1].content)
            print(json.dumps(audit, indent=2, ensure_ascii=False, default=str))

## 6. Ajouter une offre manuelle depuis le notebook

Colle une offre dans `MANUAL_DESCRIPTION`, puis execute. Si la description reste vide, la cellule ne fait rien.

In [ ]:
MANUAL_TITLE = ""
MANUAL_COMPANY = ""
MANUAL_LOCATION = "Paris, France"
MANUAL_APPLICATION_URL = ""
MANUAL_DESCRIPTION = """"""

if MANUAL_DESCRIPTION.strip():
    from smartapply.pipeline import Pipeline
    pipeline = Pipeline()
    ingest_report = pipeline.ingest_text(
        MANUAL_DESCRIPTION,
        title=MANUAL_TITLE,
        company=MANUAL_COMPANY,
        location=MANUAL_LOCATION or None,
        application_url=MANUAL_APPLICATION_URL or None,
    )
    print(json.dumps(ingest_report.__dict__, indent=2, ensure_ascii=False, default=str))
else:
    print("No manual offer provided")

## 7. Generer une candidature Autopilot pour un job precis

Mets `JOB_ID` sur un job deja analyse. Tu peux lancer `Pipeline().process_pending(...)` avant si besoin.

In [ ]:
JOB_ID = None
CREATE_GMAIL_DRAFT_FOR_JOB = False

if JOB_ID is None:
    print("Set JOB_ID to generate one application")
else:
    from smartapply.pipeline import Pipeline
    pipeline = Pipeline()
    single_report = pipeline.apply_to_autopilot(
        int(JOB_ID),
        create_gmail_draft=CREATE_GMAIL_DRAFT_FOR_JOB,
        require_quality_gate=True,
    )
    print(json.dumps(single_report.__dict__, indent=2, ensure_ascii=False, default=str))

## 8. Commandes utiles equivalentes

Ces commandes font la meme chose depuis le terminal.

In [ ]:
print("""
# Initialiser la base
.venv/bin/python -m smartapply.cli init-db

# Lancer autopilot
.venv/bin/python -m smartapply.cli autopilot \\
  --source serpapi --source francetravail --source manual \\
  --query \"Data Scientist OR Machine Learning Engineer OR AI Engineer\" \\
  --location \"Paris, France\" \\
  --target-drafts 25 \\
  --gmail-draft

# Voir les candidatures
.venv/bin/python -m smartapply.cli list-applications
""")